# RAG Pipeline — Echolabs
**Assignment 8: Rag Pipeline, System Experiments, and Failure Analysis**

In [ ]:
import re
import time
import random
import numpy as np
import pandas as pd
import torch
from typing import List, Dict
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

## Step 1.1: Define Your Knowledge Base

**Dataset:** Kaggle TED Talks transcripts, mirrored at [owentemple/TED-talks](https://github.com/owentemple/TED-talks)  

| Talk | Speaker | Keyword in URL |
|---|---|---|
| Do schools kill creativity? | Sir Ken Robinson | `robinson` |
| The power of vulnerability | Brené Brown | `brown_the_power` |
| How great leaders inspire action | Simon Sinek | `sinek` |

In [14]:
df = load_dataset("knkarthick/dialogsum", split="train")
sampled = df.select(range(40))

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [15]:
# Build the full paragraph
PARAGRAPHS = []
for i, row in enumerate(sampled):
    text = row['dialogue'].strip()
    word_count = len(text.split())
    if word_count >= 80: 
        PARAGRAPHS.append({
            'text':       text,
            'word_count': word_count,
            'speaker':    'Multiple Speakers',
            'title':      f"Dialogue {i+1}",
            'para_idx':   i,
            'summary':    row['summary']
        })

In [16]:
# Preview one paragraph
sample = PARAGRAPHS[2]
print(sample['text'])

#Person1#: Excuse me, did you see a set of keys?
#Person2#: What kind of keys?
#Person1#: Five keys and a small foot ornament.
#Person2#: What a shame! I didn't see them.
#Person1#: Well, can you help me look for it? That's my first time here.
#Person2#: Sure. It's my pleasure. I'd like to help you look for the missing keys.
#Person1#: It's very kind of you.
#Person2#: It's not a big deal.Hey, I found them.
#Person1#: Oh, thank God! I don't know how to thank you, guys.
#Person2#: You're welcome.


## Step 1.2: Text Representation & Chunking

### How Paragraphs Are Defined
Paragraphs are sentence boundary safe thought segments extracted from each talk's transcript. Stage directions are stripped first. Sentences are accumulated until the 200-word ceiling is reached; anything under 80 words is merged forward. No sentence is ever split mid-way across a boundary.

### How Overlap Is Implemented (Strategy 2)
A sliding window of size 2 advances by 1 paragraph at a time. Chunk *i* = paragraphs [*i*, *i+1*]; chunk *i+1* = paragraphs [*i+1*, *i+2*]. Paragraph *i+1* appears in both consecutive chunks — this is the overlap. With `window=2, step=1` the overlap rate is 50%. This ensures no content near a chunk boundary is lost from retrieval.

| Strategy | Unit | Description |
|---|---|---|
| **Fixed-Length** | Words | Pack sentences to a ~150-word target; sentence boundaries always respected |
| **Overlapping Paragraph** | Paragraphs | 2-paragraph sliding window, step=1; adjacent chunks share 1 paragraph (50% overlap) |
| **Hybrid/Strategic** | Talk sections | Group 3 consecutive paragraphs from the same talk into one section chunk |

In [17]:
# ----------- Strategy 1: Fixed-Length Chunking -----------

def fixed_length_chunking(paragraphs: List[Dict], target_words: int = 150) -> List[Dict]:
    full_text = ' '.join(p['text'] for p in paragraphs)
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z"])', full_text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks, current_sents, current_wc = [], [], 0

    for sent in sentences:
        sw = len(sent.split())
        if current_wc + sw > target_words and current_sents:
            chunks.append({
                'text':       ' '.join(current_sents),
                'word_count': current_wc,
                'strategy':   'fixed_length',
                'chunk_id':   len(chunks)
            })
            current_sents, current_wc = [], 0
        current_sents.append(sent)
        current_wc += sw

    if current_sents:
        chunks.append({
            'text':       ' '.join(current_sents),
            'word_count': current_wc,
            'strategy':   'fixed_length',
            'chunk_id':   len(chunks)
        })
    return chunks


fixed_chunks = fixed_length_chunking(PARAGRAPHS, target_words=150)
print(f"Fixed-Length  →  {len(fixed_chunks)} chunks")
print(f"Avg words     :  {np.mean([c['word_count'] for c in fixed_chunks]):.1f}")
print()
print("Sample chunk [0]:")
print("-" * 60)
print(fixed_chunks[0]['text'])

Fixed-Length  →  33 chunks
Avg words     :  133.9

Sample chunk [0]:
------------------------------------------------------------
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know.


In [18]:
# ── Strategy 2: Overlapping Paragraph Chunking ────────────────────────────────

def overlapping_paragraph_chunking(paragraphs: List[Dict],
                                    window: int = 2,
                                    step:   int = 1) -> List[Dict]:
    chunks = []
    for start in range(0, len(paragraphs) - window + 1, step):
        window_paras = paragraphs[start : start + window]
        combined_text = '\n\n'.join(p['text'] for p in window_paras)
        chunks.append({
            'text':            combined_text,
            'word_count':      len(combined_text.split()),
            'strategy':        'overlapping_paragraph',
            'chunk_id':        len(chunks),
            'source_para_ids': list(range(start, start + window)),
            'speaker':         window_paras[0]['speaker'],
            'title':           window_paras[0]['title'],
        })
    return chunks


overlap_chunks = overlapping_paragraph_chunking(PARAGRAPHS, window=2, step=1)
print(f"Overlapping Paragraph  →  {len(overlap_chunks)} chunks")
print(f"Avg words              :  {np.mean([c['word_count'] for c in overlap_chunks]):.1f}")
print(f"Overlap                :  1 paragraph shared per adjacent pair (50%)")
print()
print(f"Chunk 0 → paras {overlap_chunks[0]['source_para_ids']}")
print(f"Chunk 1 → paras {overlap_chunks[1]['source_para_ids']}")
print(f"  → Para 1 appears in BOTH (the overlap)")

Overlapping Paragraph  →  31 chunks
Avg words              :  274.4
Overlap                :  1 paragraph shared per adjacent pair (50%)

Chunk 0 → paras [0, 1]
Chunk 1 → paras [1, 2]
  → Para 1 appears in BOTH (the overlap)


In [25]:
# ── Strategy 3: Hybrid / Strategic Chunking ───────────────────────────────────

def hybrid_strategic_chunking(paragraphs: List[Dict],
                               paras_per_section: int = 3) -> List[Dict]:
    chunks = []
    total_sections = (len(paragraphs) + paras_per_section - 1) // paras_per_section

    for i in range(0, len(paragraphs), paras_per_section):
        section = paragraphs[i : i + paras_per_section]
        combined = '\n\n'.join(p['text'] for p in section)
        section_num = i // paras_per_section + 1
        chunks.append({
            'text':       combined,
            'word_count': len(combined.split()),
            'strategy':   'hybrid_strategic',
            'chunk_id':   len(chunks),
            'title':      f"Dialogues {i+1}–{i+len(section)}",
            'section':    f"Section {section_num}/{total_sections}",
        })
    return chunks

hybrid_chunks = hybrid_strategic_chunking(PARAGRAPHS, paras_per_section=3)
print(f"Hybrid/Strategic  →  {len(hybrid_chunks)} chunks")
print(f"Avg words         :  {np.mean([c['word_count'] for c in hybrid_chunks]):.1f}")
print()
print("Chunks by section:")
for c in hybrid_chunks:
    print(f"  [{c['chunk_id']:2d}]  {c['title']:<25}  {c['section']}  ({c['word_count']} words)")

Hybrid/Strategic  →  11 chunks
Avg words         :  401.8

Chunks by section:
  [ 0]  Dialogues 1–3              Section 1/11  (396 words)
  [ 1]  Dialogues 4–6              Section 2/11  (425 words)
  [ 2]  Dialogues 7–9              Section 3/11  (328 words)
  [ 3]  Dialogues 10–12            Section 4/11  (469 words)
  [ 4]  Dialogues 13–15            Section 5/11  (303 words)
  [ 5]  Dialogues 16–18            Section 6/11  (386 words)
  [ 6]  Dialogues 19–21            Section 7/11  (412 words)
  [ 7]  Dialogues 22–24            Section 8/11  (369 words)
  [ 8]  Dialogues 25–27            Section 9/11  (516 words)
  [ 9]  Dialogues 28–30            Section 10/11  (559 words)
  [10]  Dialogues 31–32            Section 11/11  (257 words)


## Step 1.3: Embedding Pipeline
| Size | Model | Dimensions | License |
|---|---|---|---|
| **Small** | [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | 384 | Apache 2.0 |
| **Medium** | [`sentence-transformers/all-mpnet-base-v2`](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | 768 | Apache 2.0 |
| **Large** | [`BAAI/bge-large-en-v1.5`](https://huggingface.co/BAAI/bge-large-en-v1.5) | 1024 | MIT |

In [10]:
EMBEDDING_MODELS = {
    'small':  {'hf_name': 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1',  'dims': 384},
    'medium': {'hf_name': 'sentence-transformers/multi-qa-mpnet-base-cos-v1',  'dims': 768},
    'large':  {'hf_name': 'BAAI/bge-large-en-v1.5',                   'dims': 1024},
}

loaded_embed_models = {}
for size, info in EMBEDDING_MODELS.items():
    loaded_embed_models[size] = SentenceTransformer(info['hf_name'])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
def embed_chunks(chunks: List[Dict], model: SentenceTransformer,
                 model_size: str, batch_size: int = 16) -> Dict:
    texts = [c['text'] for c in chunks]
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return {
        'embeddings': np.array(embeddings, dtype=np.float32),
        'chunks':     chunks,
        'model_size': model_size
    }


# Build index_store[strategy][model_size]
all_chunks = {
    'fixed_length':          fixed_chunks,
    'overlapping_paragraph': overlap_chunks,
    'hybrid_strategic':      hybrid_chunks,
}

index_store = {}
for strategy_name, chunks in all_chunks.items():
    index_store[strategy_name] = {}
    for size, model in loaded_embed_models.items():
        result = embed_chunks(chunks, model, size)
        index_store[strategy_name][size] = result
        print(f"  ✓ {strategy_name:<28} | {size:<6} | shape={result['embeddings'].shape}")

  ✓ fixed_length                 | small  | shape=(33, 384)
  ✓ fixed_length                 | medium | shape=(33, 768)
  ✓ fixed_length                 | large  | shape=(33, 1024)
  ✓ overlapping_paragraph        | small  | shape=(31, 384)
  ✓ overlapping_paragraph        | medium | shape=(31, 768)
  ✓ overlapping_paragraph        | large  | shape=(31, 1024)
  ✓ hybrid_strategic             | small  | shape=(11, 384)
  ✓ hybrid_strategic             | medium | shape=(11, 768)
  ✓ hybrid_strategic             | large  | shape=(11, 1024)


## Step 1.4: Retrieval System

**Flow:**
1. Encode the query with the selected embedding model
2. Compute cosine similarity between the query vector and all stored chunk vectors
3. Return the top-*k* chunks by descending similarity score

In [27]:
def retrieve(
    query:      str,
    strategy:   str = 'hybrid_strategic',
    model_size: str = 'medium',
    top_k:      int = 3
) -> List[Dict]:
    embed_model = loaded_embed_models[model_size]
    query_vec = embed_model.encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    )

    store = index_store[strategy][model_size]
    chunk_embeddings = store['embeddings']
    chunks = store['chunks']

    scores = cosine_similarity(query_vec, chunk_embeddings)[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        entry = dict(chunks[idx])
        entry['score'] = float(scores[idx])
        results.append(entry)
    return results


# ── Smoke test ────────────────────────────────────────────────────────────────
test_q = "What are people discussing about work and scheduling plans?"
results = retrieve(test_q, strategy='hybrid_strategic', model_size='medium', top_k=3)

print(f"Query: {test_q}")
print("=" * 70)
for i, r in enumerate(results):
    print(f"\nRank {i+1}  |  score={r['score']:.4f}  |  {r.get('title', r['chunk_id'])}")
    print(r['text'][:250] + '...')

Query: What are people discussing about work and scheduling plans?

Rank 1  |  score=0.2663  |  Dialogues 28–30
#Person1#: I'm tired of watching television. Let's go to cinema to- night.
#Person2#: All right. Do you want to go downtown? Or is there a good movie in the neighborhood?
#Person1#: I'd rather not spend a lot of money. What does the pa- per say about...

Rank 2  |  score=0.2272  |  Dialogues 22–24
#Person1#: Mr. White, I would like to give you notice that I will be leaving the company. It will be effective at the beginning of the next month.
#Person2#: Jessica, I am very sorry to hear that. Why are you leaving?
#Person1#: I've been offered ano...

Rank 3  |  score=0.2240  |  Dialogues 7–9
#Person1#: This is a good basic computer package. It's got a good CPU, 256 megabytes of RAM, and a DVD player.
#Person2#: Does it come with a modem?
#Person1#: Yes, it has a built-in modem. You just plug a phone line into the back of the computer.
#P...


## Step 1.5: Generation Pipeline

**Model:** `google/flan-t5-base`

In [28]:
GEN_MODEL_NAME = 'google/flan-t5-base'

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model     = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME).to(device)
gen_model.eval()

param_count = sum(p.numel() for p in gen_model.parameters())

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [29]:
def build_prompt(query: str, retrieved_chunks: List[Dict]) -> str:
    """Assemble the RAG prompt: instruction + labelled context blocks + question."""
    context_block = ''
    for i, chunk in enumerate(retrieved_chunks):
        section = chunk.get('section', f"Chunk {chunk['chunk_id']}")
        # Truncate to keep within flan-t5-base's 512-token input limit
        snippet = ' '.join(chunk['text'].split()[:250])
        context_block += f"[Context {i+1} — {section}]\n{snippet}\n\n"

    return (
        "Answer the following question using only the provided context. "
        "Be specific and concise.\n\n"
        + context_block.strip()
        + f"\n\nQuestion: {query}\nAnswer:"
    )


def generate_answer(prompt: str, max_new_tokens: int = 120) -> str:
    """Run flan-t5-base inference using beam search (matches A7 setup)."""
    inputs = gen_tokenizer(
        prompt, return_tensors='pt', max_length=512, truncation=True
    ).to(device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, num_beams=4, early_stopping=True
        )
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def rag_generate(
    query:      str,
    strategy:   str  = 'hybrid_strategic',
    model_size: str  = 'medium',
    top_k:      int  = 3,
    verbose:    bool = True
) -> Dict:
    """
    Full RAG pipeline: Query → Retrieve → Prompt → Generate.
    Returns a dict with query, strategy, model_size, retrieved_chunks,
    answer, and latency_sec — ready for the Part 2 evaluation table.
    """
    start     = time.time()
    retrieved = retrieve(query, strategy=strategy, model_size=model_size, top_k=top_k)
    prompt    = build_prompt(query, retrieved)
    answer    = generate_answer(prompt)
    latency   = round(time.time() - start, 2)

    if verbose:
        dims = EMBEDDING_MODELS[model_size]['dims']
        print(f"Query    : {query}")
        print(f"Strategy : {strategy}  |  Model: {model_size} ({dims}d)  |  Top-k: {top_k}")
        print(f"Latency  : {latency}s")
        print()
        print("Retrieved chunks:")
        for r in retrieved:
            print(f"  score={r['score']:.4f}  | {r.get('section', f'chunk_{r[chr(99)+chr(104)+chr(117)+chr(110)+chr(107)+chr(95)+chr(105)+chr(100)]}')  }")
            print(f"  Preview: {r['text'][:180]}...")
            print()
        print(f"Answer   : {answer}")

    return {
        'query':            query,
        'strategy':         strategy,
        'model_size':       model_size,
        'retrieved_chunks': retrieved,
        'answer':           answer,
        'latency_sec':      latency
    }